# The REG742_EVENT_TEXT table 

The `REG742_EVENT_TEXT` table is a essential reference (lookup) table for analyzing **Unitary Patents (UP)** in the PATSTAT EP Register database. It was introduced to support the unitary effect regulations and acts as a dedicated dictionary for legal events that occur during the unitary phase. The main role of `REG742_EVENT_TEXT` is to provide the clear text description (`EVENT_TEXT`) for the internal event codes used by the European Patent Office (EPO) to represent procedural and legal actions related to Unitary Patents. By linking this table to `REG731_EVENT_DATA`, technical event codes are translated into understandable information, allowing users to easily interpret the legal events recorded for a UP.

While `REG301_EVENT_DATA` covers legal events for all European patent applications, `REG742_EVENT_TEXT` focuses exclusively on applications that have entered the UP phase.

In [4]:
from epo.tipdata.patstat import PatstatClient
from epo.tipdata.patstat.database.models import REG731_EVENT_DATA, REG742_EVENT_TEXT
from sqlalchemy import func

# Initialise the PATSTAT client
patstat = PatstatClient(env='PROD')

# Access ORM
db = patstat.orm()

## Key fields in REG742_EVENT_TEXT table

### EVENT_CODE (primary key)
This field contains the internal code used by the EPO to identify a specific type of action, such as publications, changes or deletions, which occurs throughout the patent process. It serves as a technical identifier, allowing systems to categorize the exact nature of the procedural event taking place. The code is up to 30 characters long and may include descriptive elements, such as "9199" for actions before B1 publication or "9299" for actions after B1 publication. It helps ensure consistent identification and understanding of each specific event.

### EVENT_TEXT
This field contains the description of the action in clear text. It provides the human-readable explanation corresponding to the `EVENT_CODE`, clarifying exactly what action or event occurred.

In [2]:
q = (
    db.query(
        REG742_EVENT_TEXT.event_code,
        REG742_EVENT_TEXT.event_text
    )
    .order_by(REG742_EVENT_TEXT.event_code)
    .distinct()
)

res = patstat.df(q)
res

,event_code,event_text
0,0009700UREQ01,Filing of request for unitary effect
1,0009701UREQ02,Decision on the request for unitary effect
2,0009702UREQ10,Request for unitary effect withdrawn
3,0009703RFEE22,New entry: Payment of renewal fee
4,0009704UDLA02,Renewal fees not paid: Unitary Patent Protecti...
5,0009705LAPS22,Unitary Patent lapsed
6,0009706REES22,Request for re-establishment of rights filed
7,0009707REES26,Decision taken on request for re-establishment...
8,0009710ILIC02,New entry: Licences of right
9,0009711ILIC04,New entry: Withdrawal of statement concerning ...


## Linking REG731_EVENT_DATA to REG742_EVENT_TEXT
To enrich the information contained in `REG731_EVENT_DATA`, we can join it with the `REG742_EVENT_TEXT` reference table using the `EVENT_CODE` attribute. This join replaces an internal EPO technical code with meaningful contextual information, namely the clear text description (`EVENT_TEXT`) of the legal or procedural action.

In [5]:
q = (
    db.query(
        REG731_EVENT_DATA.id,
        REG731_EVENT_DATA.event_date,
        REG731_EVENT_DATA.event_code,
        REG742_EVENT_TEXT.event_text,
    )
    .join(
        REG742_EVENT_TEXT,
        REG742_EVENT_TEXT.event_code == REG731_EVENT_DATA.event_code
    )
    .order_by(REG731_EVENT_DATA.id)
    .distinct()
    .limit(1000)
)

res = patstat.df(q)
res

,id,event_date,event_code,event_text
0,5018904,2024-08-15,0009701UREQ02,Decision on the request for unitary effect
1,5018904,2024-06-14,0009700UREQ01,Filing of request for unitary effect
2,5701869,2023-07-21,0009700UREQ01,Filing of request for unitary effect
3,5701869,2023-10-13,0009702UREQ10,Request for unitary effect withdrawn
4,6748170,2023-10-13,0009702UREQ10,Request for unitary effect withdrawn
...,...,...,...,...
995,12182280,2024-05-10,0009700UREQ01,Filing of request for unitary effect
996,12182280,2024-08-22,0009703RFEE22,New entry: Payment of renewal fee
997,12183749,2023-10-06,0009703RFEE22,New entry: Payment of renewal fee
998,12183749,2024-09-27,0009703RFEE22,New entry: Payment of renewal fee
